## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# 50 ZipEats orders: every pair of measures plotted

**Scatter Plot Matrix** · a worked example from the StatNex teaching guide

**The story.** ZipEats is a budget food-delivery app. For 50 recent orders it has the delivery time, the distance, the order total and the tip. Before building any model, the analyst plots every pair of variables against each other.

**The question.** Do longer trips take longer, and do bigger orders earn bigger tips?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/scatter_matrix.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Delivery time (min),Distance (km),Order total ($),Tip ($)
1.0,16.0,1.5,18.41,2.07
2.0,24.0,1.8,22.84,2.68
3.0,28.0,2.4,12.55,1.77
4.0,19.0,1.0,14.25,0.0
5.0,29.0,1.9,14.64,0.21
6.0,26.0,1.9,16.77,1.34
7.0,22.0,1.2,17.94,2.51
8.0,30.0,2.9,9.43,0.0
9.0,22.0,1.3,12.85,2.08
10.0,26.0,2.1,19.0,1.79
11.0,28.0,1.2,21.85,2.97
12.0,17.0,1.6,9.97,0.0
13.0,29.0,2.5,21.83,3.46
14.0,26.0,2.3,14.55,0.89
15.0,23.0,1.3,17.27,1.06
16.0,23.0,1.5,21.22,1.76
17.0,23.0,2.2,15.01,0.95
18.0,22.0,1.8,17.59,2.45
19.0,26.0,1.4,14.39,0.86
20.0,12.0,0.7,18.94,0.64
21.0,26.0,2.5,17.0,2.43
22.0,14.0,2.0,9.05,1.5
23.0,29.0,3.2,13.19,0.0
24.0,20.0,2.3,11.02,0.99
25.0,23.0,2.1,20.25,0.63
26.0,20.0,1.3,16.26,1.03
27.0,27.0,3.2,18.54,1.01
28.0,23.0,2.2,14.43,2.45
29.0,28.0,1.8,13.04,0.67
30.0,19.0,2.7,9.3,0.76
31.0,27.0,2.3,14.7,0.02
32.0,27.0,2.5,11.43,0.98
33.0,29.0,3.1,11.19,0.55
34.0,24.0,1.8,19.65,0.81
35.0,26.0,1.6,18.99,2.62
36.0,18.0,1.1,20.07,0.83
37.0,25.0,2.2,12.95,2.23
38.0,58.0,2.0,15.34,1.43
39.0,17.0,1.0,10.77,0.0
40.0,31.0,2.6,20.83,0.22
41.0,27.0,2.2,13.66,1.75
42.0,18.0,1.3,12.59,0.42
43.0,24.0,1.4,11.63,0.55
44.0,29.0,1.6,12.33,1.37
45.0,25.0,3.1,12.61,0.74
46.0,24.0,1.2,14.06,1.97
47.0,25.0,2.3,19.86,1.34
48.0,23.0,2.6,12.98,2.01
49.0,20.0,0.7,14.33,0.7
50.0,26.0,1.6,9.85,1.46
""")

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 160)

## The data

In [ ]:
# One column per variable, one row per case. A case with a blank or non-numeric
# value in any of the variables is dropped from all of them.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
numeric = data[columns].apply(pd.to_numeric, errors="coerce").dropna()

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1. Kurtosis and skewness are Excel's KURT and SKEW,
# the sample-corrected forms (bias=False): both are 0 for a normal distribution.
# Skewness above about 1 means a long right tail; a large kurtosis means heavy tails,
# and one extreme value is enough to produce it. (The normal probability plot reports
# the plain moment ratios instead, with kurtosis 3 for a normal distribution.)
def describe(x):
    return {"n": x.size, "mean": np.mean(x), "variance": np.var(x, ddof=1),
            "sd": np.std(x, ddof=1), "min": x.min(), "max": x.max(),
            "kurtosis": stats.kurtosis(x, bias=False), "skewness": stats.skew(x, bias=False)}


summary = pd.DataFrame({name: describe(numeric[name].to_numpy(float)) for name in columns})
print(summary)

## Correlation matrix

In [ ]:
# Pearson's r for each pair: the strength of the straight-line link, from -1 to +1.
# No P values are printed; one unusual case can raise or lower r a lot, which only
# the plots below reveal.
correlation = pd.DataFrame(np.corrcoef(numeric.to_numpy(float), rowvar=False),
                           index=columns, columns=columns)
print()
print(correlation)

## Scatter plot matrix

In [ ]:
# Every pair of variables at once: curves, clusters and outliers show here and in
# none of the numbers above.
Path("figures").mkdir(exist_ok=True)
axes = pd.plotting.scatter_matrix(numeric, figsize=(2.2 * len(columns), 2.2 * len(columns)))
plt.savefig("figures/scatter_matrix.png", dpi=150)
plt.close("all")

## What the result says

A scatter plot matrix draws every pair of measures against each other. The correlation matrix gives one number for each plot. A correlation near 0 means no straight-line link.

Order total and tip have a correlation of 0.43: bigger orders tend to earn bigger tips. Delivery time and distance have 0.38: longer trips tend to take longer. Both links are moderate.

Look at the plot of delivery time against distance. One order took 58 minutes for a trip of only 2 km, and it sits far from all the other points. This order also explains the high skewness (a long tail on one side) of delivery time, 2.58. It pulls the correlation down, so the trip link is really stronger than 0.38 suggests.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.